# Module 14: Interactive Pydantic V2 & High-Speed Data Modeling

### What You Will Discover
By running this notebook, you will explore Pydantic V2's compiled Rust engine (`pydantic-core`), implement `@field_validator` and `@model_validator` rules, and parse polymorphic events using **Discriminated Unions** in $O(1)$ time.

**Key Question Answered:** *Why are tagged Discriminated Unions 10x faster and safer than naive `Union[A, B, C]` types?*


In [ ]:
# Step 1: Base Pydantic V2 Model
from pydantic import BaseModel, Field


class User(BaseModel):
    user_id: int
    username: str = Field(min_length=3, max_length=20)
    email: str
    is_active: bool = True


In [ ]:
# Step 2: Instantiating and validating
u = User(user_id=1, username='karthik', email='karthik@example.com')
print(f'User object: {u}')
print(f'Serialized dict: {u.model_dump()}')


In [ ]:
# Step 3: Pydantic V2 field validator
from pydantic import field_validator


class ValidatedUser(User):
    @field_validator('username')
    @classmethod
    def normalize_username(cls, val: str) -> str:
        return val.strip().lower()

vu = ValidatedUser(user_id=2, username='  ALICE  ', email='alice@site.com')
print(f'Normalized username: "{vu.username}"')


### 🔮 Prediction Prompt
**Before running the next cell:** In Pydantic V2, what happens if you pass a float `19.99` to a field annotated as `int` in default loose mode vs `strict=True` mode? Write down your prediction.


In [ ]:
# Surprising Result: Strict Mode Rejects Implicit Lossy Coercions
from pydantic import ValidationError


class StrictQuantity(BaseModel):
    quantity: int = Field(strict=True)

try:
    StrictQuantity(quantity=19.99)  # In V1 or loose mode, this might truncate to 19!
except ValidationError as exc:
    print(f'Strict mode caught lossy coercion:\n{exc.errors()[0]["msg"]}')
    print('Explanation: strict=True prevents silent float-to-int truncation bugs!')


### Discriminated Unions: $O(1)$ Direct Dispatch
Tagging polymorphic models with a discriminator field eliminates trial-and-error union evaluation.


In [ ]:
from typing import Annotated, Literal

from pydantic import TypeAdapter


class CreditCard(BaseModel):
    payment_type: Literal['credit_card']
    last4: str

class Crypto(BaseModel):
    payment_type: Literal['crypto']
    wallet_address: str

PaymentMethod = Annotated[CreditCard | Crypto, Field(discriminator='payment_type')]
adapter = TypeAdapter(PaymentMethod)

payment = adapter.validate_python({'payment_type': 'crypto', 'wallet_address': '0x71C...'})
print(f'Directly parsed type: {type(payment).__name__}, wallet: {payment.wallet_address}')


### Model Validators: Cross-Field Invariant Checking
Validate relationships across multiple fields with `@model_validator(mode='after')`.


In [ ]:
from pydantic import model_validator


class DateRange(BaseModel):
    start_day: int
    end_day: int

    @model_validator(mode='after')
    def check_range(self) -> 'DateRange':
        if self.end_day <= self.start_day:
            raise ValueError('end_day must be strictly greater than start_day')
        return self

dr = DateRange(start_day=1, end_day=10)
print(f'Valid range: {dr.start_day} -> {dr.end_day}')


### 🛠️ Interactive Challenge: Fix Deprecated V1 Validator Syntax
The following code uses deprecated Pydantic V1 syntax (`@validator` and `.dict()`). Fix it to use modern Pydantic V2 (`@field_validator` with `@classmethod` and `.model_dump()`).


In [ ]:
# TODO: FIX ME - Update to modern Pydantic V2 syntax
class Account(BaseModel):
    account_id: str

    # FIX: Replace @validator with @field_validator and add @classmethod
    @field_validator('account_id')
    @classmethod
    def check_prefix(cls, v: str) -> str:
        if not v.startswith('acc_'):
            raise ValueError('Account ID must start with acc_')
        return v

acc = Account(account_id='acc_9921')
# FIX: Use .model_dump() instead of deprecated .dict()
print(f'Validated account dictionary: {acc.model_dump()}')


### 🏁 Summary & Next Steps
- Pydantic V2 is powered by Rust (`pydantic-core`) for 5-20x speedups.
- Use `@field_validator` for single attributes; `@model_validator` for inter-field rules.
- Always use Discriminated Unions for polymorphic schemas.
- Run `python 01_pydantic_validators_demo.py` and `python 02_discriminated_unions_and_polymorphism_demo.py`.
- Follow [PROJECT_GUIDE.md](PROJECT_GUIDE.md) to implement the validation engine.
